In [0]:
import requests

DISCOVERY_URL = "https://sharedmobility.ch/gbfs.json"   # GBFS 2.0, no key needed

r = requests.get(DISCOVERY_URL, timeout = 30)
r.raise_for_status()
disc = r.json()

lang = next(iter(disc["data"]))
feeds = {f["name"]: f["url"] for f in disc["data"][lang]["feeds"] }
feeds

{'providers': 'https://sharedmobility.ch/providers.json',
 'system_information': 'https://sharedmobility.ch/system_information.json',
 'station_information': 'https://sharedmobility.ch/station_information.json',
 'free_bike_status': 'https://sharedmobility.ch/free_bike_status.json',
 'station_status': 'https://sharedmobility.ch/station_status.json',
 'system_hours': 'https://sharedmobility.ch/system_hours.json',
 'system_regions': 'https://sharedmobility.ch/system_regions.json',
 'system_pricing_plans': 'https://sharedmobility.ch/system_pricing_plans.json',
 'geofencing_zones': 'https://www.uvek-gis.admin.ch/BFE/disharing/geofences/geofencing_zones.json'}

In [0]:
import json
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

VOL = "/Volumes/workspace/ex3_sdp_bike/bikeflow"
NEEDED_DATA = ["station_status", "station_information"]

ts = datetime.now(ZoneInfo("Europe/Zurich")).strftime("%Y-%m-%d-%H-%M-%SZ")

print(f"Timestamp: {ts}")

for name in NEEDED_DATA:
    payload = requests.get(feeds[name], timeout=30).json()
    folder = f"{VOL}/{name}"
    dbutils.fs.mkdirs(folder)
    with open (f"{folder}/{name}_{ts}.json", "w") as fh:
        json.dump(payload, fh)
    print(name, len(payload["data"]["stations"]), "stations ->", folder)

station_status 12335 stations -> /Volumes/workspace/ex3_sdp_bike/bikeflow/station_status
station_information 12679 stations -> /Volumes/workspace/ex3_sdp_bike/bikeflow/station_information


In [0]:
%sql
SELECT 
    _metadata.file_name,
    from_utc_timestamp(TIMESTAMP(last_updated), "Europe/Zurich") as last_updated_ts,
    SIZE(data.stations) AS n_stations
FROM read_files(
    '/Volumes/workspace/ex3_sdp_bike/bikeflow/station_status/',
    format => 'json');

file_name,last_updated_ts,n_stations
station_status_2026-10-13-22-57-18Z.json,2026-10-13T22:57:19.000Z,12335
